# Subject-Holdout Generalization Test — ISOT

Tests whether the models can generalize to subject categories they've never seen during training.

- **Train on:** politicsNews, News, left-news, politics (mix of real + fake subjects)
- **Test on:** worldnews, US_News, Government News, Middle-east (held-out subjects)

If accuracy drops significantly compared to the original random-split results, that confirms the models were relying on subject-specific writing patterns rather than genuine fake-vs-real understanding.

## 1. Load Data and Build Subject-Holdout Split


In [1]:
import pandas as pd

train_df_orig = pd.read_csv("train.csv")
val_df_orig = pd.read_csv("val.csv")
test_df_orig = pd.read_csv("test.csv")

full_df = pd.concat([train_df_orig, val_df_orig, test_df_orig], ignore_index=True)
print("Full dataset shape:", full_df.shape)
print("\nSubject counts:")
print(full_df["subject"].value_counts())


Full dataset shape: (38635, 8)

Subject counts:
subject
politicsNews       11208
worldnews           9983
News                9050
politics            4332
left-news           2411
Government News      868
Middle-east          400
US_News              383
Name: count, dtype: int64


In [2]:
TRAIN_SUBJECTS = ["politicsNews", "News", "left-news", "politics"]
TEST_SUBJECTS = ["worldnews", "US_News", "Government News", "Middle-east"]

holdout_train_df = full_df[full_df["subject"].isin(TRAIN_SUBJECTS)].reset_index(drop=True)
holdout_test_df = full_df[full_df["subject"].isin(TEST_SUBJECTS)].reset_index(drop=True)

print(f"Holdout train set: {len(holdout_train_df)} rows")
print(holdout_train_df.groupby(["subject", "label"]).size())

print(f"\nHoldout test set: {len(holdout_test_df)} rows")
print(holdout_test_df.groupby(["subject", "label"]).size())

print(f"\nTrain label balance: {holdout_train_df['label'].value_counts(normalize=True).to_dict()}")
print(f"Test label balance: {holdout_test_df['label'].value_counts(normalize=True).to_dict()}")


Holdout train set: 27001 rows
subject       label
News          fake      9050
left-news     fake      2411
politics      fake      4332
politicsNews  real     11208
dtype: int64

Holdout test set: 11634 rows
subject          label
Government News  fake      868
Middle-east      fake      400
US_News          fake      383
worldnews        real     9983
dtype: int64

Train label balance: {'fake': 0.5849042628050813, 'real': 0.4150957371949187}
Test label balance: {'real': 0.85808836169847, 'fake': 0.14191163830153}


## 2. TF-IDF Baseline on Subject-Holdout Split


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

holdout_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        stop_words="english",
        min_df=2
    )),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

holdout_pipeline.fit(holdout_train_df["text"], holdout_train_df["label_id"])

holdout_preds = holdout_pipeline.predict(holdout_test_df["text"])
holdout_acc = accuracy_score(holdout_test_df["label_id"], holdout_preds)
holdout_prec, holdout_rec, holdout_f1, _ = precision_recall_fscore_support(
    holdout_test_df["label_id"], holdout_preds, average="binary"
)

print(f"Subject-holdout Test Accuracy: {holdout_acc:.4f}")
print(f"Precision: {holdout_prec:.4f}, Recall: {holdout_rec:.4f}, F1: {holdout_f1:.4f}")
print()
print(classification_report(holdout_test_df["label_id"], holdout_preds, target_names=["real", "fake"]))


Subject-holdout Test Accuracy: 0.9350
Precision: 0.7022, Recall: 0.9412, F1: 0.8043

              precision    recall  f1-score   support

        real       0.99      0.93      0.96      9983
        fake       0.70      0.94      0.80      1651

    accuracy                           0.94     11634
   macro avg       0.85      0.94      0.88     11634
weighted avg       0.95      0.94      0.94     11634



## 3. Compare Against Original Random-Split Accuracy


In [4]:
original_comparison = pd.read_csv("full_model_comparison.csv")
original_tfidf_acc = original_comparison[
    original_comparison["model"] == "TF-IDF + Logistic Regression"
]["test_accuracy"].values[0]

print(f"Original (random split) TF-IDF accuracy: {original_tfidf_acc:.4f}")
print(f"Subject-holdout TF-IDF accuracy:          {holdout_acc:.4f}")
print(f"Drop: {original_tfidf_acc - holdout_acc:.4f} ({(original_tfidf_acc - holdout_acc)*100:.1f} points)")


Original (random split) TF-IDF accuracy: 0.9860
Subject-holdout TF-IDF accuracy:          0.9350
Drop: 0.0510 (5.1 points)


## 4. DistilBERT on Subject-Holdout Split


In [5]:
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split as sk_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Small validation split carved out of the holdout training subjects (for early stopping)
holdout_train_split, holdout_val_split = sk_split(
    holdout_train_df, test_size=0.1, random_state=42, stratify=holdout_train_df["label_id"]
)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 256
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[["text", "label_id"]].rename(columns={"label_id": "labels"}).reset_index(drop=True)
    )

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = to_hf_dataset(holdout_train_split).map(tokenize_fn, batched=True)
val_ds = to_hf_dataset(holdout_val_split).map(tokenize_fn, batched=True)
test_ds = to_hf_dataset(holdout_test_df).map(tokenize_fn, batched=True)

for ds in [train_ds, val_ds, test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1}

model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

training_args = TrainingArguments(
    output_dir="./distilbert_subject_holdout",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


Using device: cuda


Map:   0%|          | 0/24300 [00:00<?, ? examples/s]

Map:   0%|          | 0/2701 [00:00<?, ? examples/s]

Map:   0%|          | 0/11634 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000102,0.000064,1.000000,1.000000,1.000000,1.000000
2,0.000033,0.001189,0.999630,0.999367,1.000000,0.999684
3,0.000014,0.000008,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4557, training_loss=0.022388677347664455, metrics={'train_runtime': 1079.024, 'train_samples_per_second': 112.602, 'train_steps_per_second': 7.039, 'total_flos': 4828436681011200.0, 'train_loss': 0.022388677347664455, 'epoch': 3.0})

In [6]:
test_output = trainer.predict(test_ds)
test_preds = test_output.predictions.argmax(axis=-1)
test_labels = test_output.label_ids

distilbert_holdout_acc = accuracy_score(test_labels, test_preds)
print(f"DistilBERT subject-holdout Test Accuracy: {distilbert_holdout_acc:.4f}")
print(classification_report(test_labels, test_preds, target_names=["real", "fake"]))

original_distilbert_acc = original_comparison[
    original_comparison["model"] == "DistilBERT"
]["test_accuracy"].values[0]

print(f"\nOriginal (random split) DistilBERT accuracy: {original_distilbert_acc:.4f}")
print(f"Subject-holdout DistilBERT accuracy:          {distilbert_holdout_acc:.4f}")
print(f"Drop: {original_distilbert_acc - distilbert_holdout_acc:.4f} ({(original_distilbert_acc - distilbert_holdout_acc)*100:.1f} points)")


DistilBERT subject-holdout Test Accuracy: 0.9993
              precision    recall  f1-score   support

        real       1.00      1.00      1.00      9983
        fake       1.00      1.00      1.00      1651

    accuracy                           1.00     11634
   macro avg       1.00      1.00      1.00     11634
weighted avg       1.00      1.00      1.00     11634


Original (random split) DistilBERT accuracy: 0.9997
Subject-holdout DistilBERT accuracy:          0.9993
Drop: 0.0003 (0.0 points)


## 5. RoBERTa on Subject-Holdout Split

In [7]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification

ROBERTA_MODEL_NAME = "roberta-base"
roberta_tokenizer = RobertaTokenizerFast.from_pretrained(ROBERTA_MODEL_NAME)

def roberta_tokenize_fn(batch):
    return roberta_tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

roberta_train_ds = to_hf_dataset(holdout_train_split).map(roberta_tokenize_fn, batched=True)
roberta_val_ds = to_hf_dataset(holdout_val_split).map(roberta_tokenize_fn, batched=True)
roberta_test_ds = to_hf_dataset(holdout_test_df).map(roberta_tokenize_fn, batched=True)

for ds in [roberta_train_ds, roberta_val_ds, roberta_test_ds]:
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

roberta_model = RobertaForSequenceClassification.from_pretrained(ROBERTA_MODEL_NAME, num_labels=2)
roberta_model.to(device)

roberta_training_args = TrainingArguments(
    output_dir="./roberta_subject_holdout",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    save_total_limit=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    report_to="none"
)

roberta_trainer = Trainer(
    model=roberta_model, args=roberta_training_args,
    train_dataset=roberta_train_ds, eval_dataset=roberta_val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

roberta_trainer.train()


Map:   0%|          | 0/24300 [00:00<?, ? examples/s]

Map:   0%|          | 0/2701 [00:00<?, ? examples/s]

Map:   0%|          | 0/11634 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6076, training_loss=0.02252056881177795, metrics={'train_runtime': 2808.5889, 'train_samples_per_second': 43.26, 'train_steps_per_second': 2.704, 'total_flos': 1.2787197290496e+16, 'train_loss': 0.02252056881177795, 'epoch': 4.0})

In [8]:
roberta_test_output = roberta_trainer.predict(roberta_test_ds)
roberta_test_preds = roberta_test_output.predictions.argmax(axis=-1)
roberta_test_labels = roberta_test_output.label_ids

roberta_holdout_acc = accuracy_score(roberta_test_labels, roberta_test_preds)
print(f"RoBERTa subject-holdout Test Accuracy: {roberta_holdout_acc:.4f}")
print(classification_report(roberta_test_labels, roberta_test_preds, target_names=["real", "fake"]))

original_roberta_acc = original_comparison[
    original_comparison["model"] == "RoBERTa"
]["test_accuracy"].values[0]

print(f"\nOriginal (random split) RoBERTa accuracy: {original_roberta_acc:.4f}")
print(f"Subject-holdout RoBERTa accuracy:          {roberta_holdout_acc:.4f}")
print(f"Drop: {original_roberta_acc - roberta_holdout_acc:.4f} ({(original_roberta_acc - roberta_holdout_acc)*100:.1f} points)")


RoBERTa subject-holdout Test Accuracy: 0.9992
              precision    recall  f1-score   support

        real       1.00      1.00      1.00      9983
        fake       1.00      1.00      1.00      1651

    accuracy                           1.00     11634
   macro avg       1.00      1.00      1.00     11634
weighted avg       1.00      1.00      1.00     11634


Original (random split) RoBERTa accuracy: 0.9988
Subject-holdout RoBERTa accuracy:          0.9992
Drop: -0.0004 (-0.0 points)


## 6. Full Three-Model Summary


In [9]:
summary = pd.DataFrame([
    {"model": "TF-IDF + Logistic Regression", "original_accuracy": original_tfidf_acc, "subject_holdout_accuracy": holdout_acc},
    {"model": "DistilBERT", "original_accuracy": original_distilbert_acc, "subject_holdout_accuracy": distilbert_holdout_acc},
    {"model": "RoBERTa", "original_accuracy": original_roberta_acc, "subject_holdout_accuracy": roberta_holdout_acc},
])
summary["accuracy_drop"] = summary["original_accuracy"] - summary["subject_holdout_accuracy"]
summary.to_csv("subject_holdout_comparison.csv", index=False)
summary


,model,original_accuracy,subject_holdout_accuracy,accuracy_drop
0,TF-IDF + Logistic Regression,0.986025,0.935018,0.051007
1,DistilBERT,0.999655,0.999312,0.000343
2,RoBERTa,0.998792,0.999226,-0.000434
